# Benchmark: the Strands decider 2B on the same tests as Signpost

**Part 4 of "LLMs the Hard Way"** (branch `signpost`), next to `03_lora_signpost.ipynb`. [`StrandsAgents/strands-decider-2B-hobson-v19`](https://huggingface.co/StrandsAgents/strands-decider-2B-hobson-v19) is a ready-made **decision model**: Qwen3.5-2B-Base with a LoRA adapter and a small readout head. Given a *state* (text) and a question with named options, it returns a calibrated probability per option. Routing is exactly such a choice: the state is the conversation, and the options are the agents with their descriptions.

**The rule:** both models get **exactly the same test items**; only the presentation differs (each model gets its own input format). The decider is used as published, zero-shot, with nothing trained here.

| Test | What it contains | Items |
|---|---|---|
| E11 | One opening message, 5 agents | 100 messages, 10 companies |
| E10 | Realistic chat history: route each user turn live | 35 conversations, 193 turns, 13 companies |
| Banking | Banking chat history (benchmark + extended), 5 agents | 225 conversations |
| E12 | Grounded questions in the decider's native types: yes / no, choice, score | 170 |
| Independent | Written outside the project: questions, routing, first messages | 300 · 172 turns · 200 |

Sections 1–3 load everything; the tests can then run in any order (section 12 needs section 10's `ask_item`). Section 11 measures latency the same way as `03`.

**Runs on:** a Kaggle GPU. On a 2-GPU Kaggle machine it uses the second GPU (`cuda:1`), so it can run next to `03`'s training. Set `DECIDER_DEVICE` to choose.

## 1 — Setup

Install the decider package, pin transformers to the same version as `03_lora_signpost.ipynb`, and show the GPU. Kaggle's preinstalled `torchao` is removed first: the current `peft` rejects it.

In [ ]:
!pip uninstall -y -q torchao
!pip install -q "strands-decider" "transformers==5.17.0"
import os

import torch

DECIDER_DEVICE = os.environ.get("DECIDER_DEVICE", "cuda:1" if torch.cuda.device_count() > 1 else "cuda:0")
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
      "| bf16 supported:", torch.cuda.is_available() and torch.cuda.is_bf16_supported(), "| decider on", DECIDER_DEVICE)

## 2 — Test data

The test files live in the repo on the experiment branch. They are cloned (or refreshed) from that branch.

In [ ]:
import json, subprocess, time
from collections import Counter
from pathlib import Path

DATA_BRANCH = "signpost"                                    # the branch of 03 and this notebook
repo = Path("llms-the-hard-way")
if not repo.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", DATA_BRANCH,
                    "https://github.com/skamalj/llms-the-hard-way.git", str(repo)], check=True)
else:
    subprocess.run(["git", "-C", str(repo), "fetch", "--depth", "1", "origin", DATA_BRANCH], check=True)
    subprocess.run(["git", "-C", str(repo), "checkout", "-B", DATA_BRANCH, "FETCH_HEAD"], check=True)
DATA_DIR = repo / "data" / "modernbert_router"

ft = json.load(open(DATA_DIR / "first_turn" / "messages.json"))
real_convs = json.load(open(DATA_DIR / "realistic" / "conversations.json"))
banking_all = json.load(open(DATA_DIR / "banking" / "benchmark.json")) + json.load(open(DATA_DIR / "banking" / "extended.json"))
BANKING_AGENTS = {                                          # exactly as in 02_frozen_router.ipynb, Cell 37, and 03
    "Payments Agent": "Handles bank transfers, failed or pending payments, transaction status, payment charges and "
                      "duplicate charges, refunds for transactions, and settlement problems where money was sent but "
                      "not received. Does not handle card servicing such as blocking, replacing or activating cards.",
    "Card Support Agent": "Handles debit and credit card servicing: card activation, blocking and unblocking, lost or "
                          "stolen cards, card replacement and delivery, card limits, card PIN, card security and "
                          "compromised card details, and card-specific transactions.",
    "Investment Agent": "Handles mutual funds, SIPs, investments, portfolio allocation, investment transactions, "
                        "redemptions and investment planning.",
    "Insurance Agent": "Handles insurance policies, premiums, renewals, claims, policy expiry and policy servicing.",
    "General Customer Support Agent": "Handles general account questions, profile and contact details, issues that do "
                                      "not clearly belong to a specialist, and routing customers to the right agent.",
}
print(f"E11 first turn: {len(ft['messages'])} messages | E10 realistic: {len(real_convs)} conversations, "
      f"{sum(len(c['user_turns']) for c in real_convs)} turns | banking: {len(banking_all)} conversations")

## 3 — Load the decider

`load_engine` downloads the base model (Qwen3.5-2B-Base) and the checkpoint's adapter and head. The checkpoint asks for **bfloat16**. A T4 has no bfloat16 support, so in that case the model is moved to **float32** (about 8 GB for 2B parameters). A first question from the model card checks that it answers sensibly.

In [ ]:
from strands_decider.infer import load_engine
from strands_decider.schema import ChoiceQuestion

CHECKPOINT = "StrandsAgents/strands-decider-2B-hobson-v19"
t0 = time.perf_counter()
engine = load_engine(CHECKPOINT, device=DECIDER_DEVICE)
if not torch.cuda.is_bf16_supported():                       # e.g. a T4: run in float32 instead of bfloat16
    engine.model.to(torch.float32)
print(f"loaded {CHECKPOINT} in {time.perf_counter() - t0:.0f}s | dtype {next(engine.model.torso.parameters()).dtype}")

demo = engine.ask("Help! My payouts have been failing for 3 days!",
                  {"team": ChoiceQuestion(instructions="Which team should handle this?",
                                          criteria={"billing": "billing", "sales": "sales", "retail": "retail"})})
print("model card example ->", demo.answers["team"].choice, demo.answers["team"].probabilities)

## 4 — The routing question

- **State:** the conversation as a plain transcript, one line per message, with the latest user message repeated at the end so the question is unambiguous.
- **Question:** *"Which agent should handle the user's latest message?"*
- **Options:** each agent name with its description.

`window` keeps only the last N user turns (with the replies between them), to compare full history with a recent window.

In [ ]:
INSTRUCTIONS = "Which agent should handle the user's latest message?"


def state_text(messages, window=None):
    """Transcript of the conversation up to the latest user message, optionally only the last `window` user turns."""
    if window is not None:
        user_idx = [i for i, m in enumerate(messages) if m["role"] == "user"]
        messages = messages[user_idx[-window]:] if len(user_idx) > window else messages
    lines = [f"{m['role']}: {m['content']}" for m in messages]
    return "Conversation:\n" + "\n".join(lines) + f"\n\nLatest user message: {messages[-1]['content']}"


def decide(messages, agents, window=None):
    """-> (chosen agent, probabilities {agent: p}) for the latest user message."""
    q = ChoiceQuestion(instructions=INSTRUCTIONS, criteria=dict(agents))
    ans = engine.ask(state_text(messages, window), {"route": q}).answers["route"]
    probs = dict(sorted(ans.probabilities.items(), key=lambda kv: -kv[1]))
    return ans.choice, probs


print(state_text(real_convs[0]["messages"][:5]))
print(decide(real_convs[0]["messages"][:5], real_convs[0]["agents"]))

## 5 — E11: first message only (100 messages, 5 agents each)

Top-1 and top-2 per message type, and every mistake. Signpost (`03`, run 25) scored **97 / 100** top-1.

In [ ]:
t0 = time.perf_counter()
e11 = []
for m in ft["messages"]:
    agents = ft["companies"][m["company"]]
    pick, probs = decide([{"role": "user", "content": m["message"]}], agents)
    e11.append({**m, "pick": pick, "top2": list(probs)[:2], "p": probs[pick]})
print(f"E11: {len(e11)} decisions in {time.perf_counter() - t0:.0f}s\n")

print(f"{'type':10s} | {'msgs':>4} | {'top-1':>9} | {'top-2':>9}")
for tt in ["clear", "short", "indirect", "close", "noisy", "ALL"]:
    rows = [r for r in e11 if tt == "ALL" or r["type"] == tt]
    print(f"{tt:10s} | {len(rows):>4} | {sum(r['pick'] == r['target_agent'] for r in rows):>4} / {len(rows):<3} | "
          f"{sum(r['target_agent'] in r['top2'] for r in rows):>4} / {len(rows):<3}")
print("\nmistakes:")
for r in [r for r in e11 if r["pick"] != r["target_agent"]]:
    print(f"  {r['type']:8s} | {r['message'][:60]:60s} | expected {r['target_agent'].replace(' Agent', ''):26s} | got {r['pick'].replace(' Agent', '')} ({r['p']:.2f})")

## 6 — E10: realistic chat history (193 turns)

Every user turn is routed with the conversation so far, assistant replies included, in two variants:
- **full** history;
- a **window** of the last 3 user turns.

The decider has no "current agent" input, so each turn is routed independently. Signpost (`03`, run 25), which sees the full history and its own previous pick as the current agent, scored **181 / 193** (78 / 83 at the test companies).

In [ ]:
TURN_TYPES = ["start", "answer", "follow", "aspect", "closing", "also_stay", "switch", "short_switch", "return", "handoff_ok", "branch"]
t0 = time.perf_counter()
e10 = []
for conv in real_convs:
    for u in conv["user_turns"]:
        prefix = conv["messages"][:u["message_index"] + 1]
        full, _ = decide(prefix, conv["agents"])
        win, _ = decide(prefix, conv["agents"], window=3)
        e10.append({"id": conv["id"], "type": u["turn_type"], "expected": u["expected_agent"],
                    "message": prefix[-1]["content"], "full": full, "window 3": win})
print(f"E10: {len(e10)} turns x 2 variants in {time.perf_counter() - t0:.0f}s\n")

print(f"{'turn type':14s} | {'turns':>5} | {'full history':>12} | {'window 3':>10}")
for tt in TURN_TYPES + ["ALL"]:
    rows = [r for r in e10 if tt == "ALL" or r["type"] == tt]
    if rows:
        print(f"{tt:14s} | {len(rows):>5} | {sum(r['full'] == r['expected'] for r in rows):>6} / {len(rows):<4} | "
              f"{sum(r['window 3'] == r['expected'] for r in rows):>5} / {len(rows):<4}")
print("\nfull-history mistakes (first 15):")
for r in [r for r in e10 if r["full"] != r["expected"]][:15]:
    print(f"  {r['id']:9s} {r['type']:12s} | {r['message'][:50]:50s} | expected {r['expected'].replace(' Agent', ''):22s} | got {r['full'].replace(' Agent', '')}")

## 7 — Banking chat history (225 conversations, 5 agents)

The banking test from `02` and `03` (benchmark + extended), with the last user message routed. Signpost (run 25) scored **210 / 225** (benchmark **48 / 50**). The decider's training data includes banking77 and CLINC-OOS, so this set isn't unseen for it.

In [ ]:
t0 = time.perf_counter()
bank = []
for ex in banking_all:
    pick, _ = decide(ex["messages"], BANKING_AGENTS)
    bank.append(pick == ex["target_agent"])
print(f"banking: {sum(bank[:50])} / 50 benchmark | {sum(bank)} / {len(bank)} all | {time.perf_counter() - t0:.0f}s")

## 8 — Summary next to Signpost

Signpost's numbers are from `03_lora_signpost.ipynb` (run 25, the published model). For E10, the decider is shown with the full history (what Signpost sees) and with a window of the last 3 user turns.

In [ ]:
signpost = {"E11 first turn, top-1 (100)": 97, "E10 chat history, all (193)": 181, "banking benchmark (50)": 48, "all banking (225)": 210}
dec_full = {"E11 first turn, top-1 (100)": sum(r["pick"] == r["target_agent"] for r in e11),
            "E10 chat history, all (193)": sum(r["full"] == r["expected"] for r in e10),
            "banking benchmark (50)": sum(bank[:50]), "all banking (225)": sum(bank)}
dec_w3 = {"E10 chat history, all (193)": sum(r["window 3"] == r["expected"] for r in e10)}
print(f"{'test':32s} | {'Signpost (run 25)':>17} | {'decider, full':>13} | {'decider, last 3':>15}")
for k in signpost:
    print(f"{k:32s} | {signpost[k]:>17} | {dec_full[k]:>13} | {dec_w3.get(k, '—'):>15}")

In [ ]:
# Section 9 — E10 on the split of 02_frozen_router.ipynb Cell 70 and 03: 6 test companies (83 turns), by turn type

companies = sorted({c["company"] for c in real_convs})
TEST_COMPANIES = set(companies) - set(companies[::2])          # identical split to 02 Cell 70 and 03
company_of = {c["id"]: c["company"] for c in real_convs}
test_rows = [r for r in e10 if company_of[r["id"]] in TEST_COMPANIES]

print(f"TEST companies: {', '.join(sorted(TEST_COMPANIES))} | {len(test_rows)} turns")
print(f"{'turn type':14s} | {'turns':>5} | {'decider full':>12} | {'decider window 3':>16}")
for tt in TURN_TYPES + ["ALL"]:
    rows = [r for r in test_rows if tt == "ALL" or r["type"] == tt]
    if rows:
        print(f"{tt:14s} | {len(rows):>5} | {sum(r['full'] == r['expected'] for r in rows):>7} / {len(rows):<3} | "
              f"{sum(r['window 3'] == r['expected'] for r in rows):>10} / {len(rows):<3}")


## 10 — E12: grounded questions

**What.** The decider on its home ground: general questions about a given text, in its **native** question types. These are the same 170 items as `02` Cell 72 and `03` (`data/modernbert_router/choice_test/items.json`). Every answer is in the text; none needs world knowledge.

| E12 type | Decider question | Read back as |
|---|---|---|
| yes / no (58) | `NoulQuestion(instructions=question)` | P(true) > 0.5 → "yes" |
| choice (59) | `ChoiceQuestion(instructions=question, criteria={option: option})`, as in the model-card example | the chosen option |
| score (53) | `ScoreQuestion(instructions=question, criteria=levels from low to high)` | the most likely level, and the expected level (its `score`) |

**Expect.** Pass / total per type and per category, score errors in levels, the question-group check (same text, different questions), every mistake, and the time per question. It runs after sections 1–3 only (setup, data, load).


In [ ]:
# Section 10 — E12 with the decider in its native question types
from strands_decider.schema import NoulQuestion, ScoreQuestion

e12_items = json.load(open(DATA_DIR / "choice_test" / "items.json"))["items"]


def ask_item(it):
    """-> (pick, probabilities in option order, expected level or None)."""
    if it["type"] == "noul":
        p_true = engine.ask(it["state"], {"q": NoulQuestion(instructions=it["question"])}).answers["q"].noul
        return ("yes" if p_true > 0.5 else "no"), [p_true, 1 - p_true], None
    if it["type"] == "choice":
        ans = engine.ask(it["state"], {"q": ChoiceQuestion(instructions=it["question"],
                                                            criteria={o: o for o in it["options"]})}).answers["q"]
        return ans.choice, [ans.probabilities.get(o, 0.0) for o in it["options"]], None
    ans = engine.ask(it["state"], {"q": ScoreQuestion(instructions=it["question"], criteria=it["options"])}).answers["q"]
    # probabilities are keyed by level name or by level index, depending on the version
    probs = [ans.probabilities.get(o, ans.probabilities.get(str(i), 0.0)) for i, o in enumerate(it["options"])]
    top = max(range(len(probs)), key=lambda i: probs[i]) if any(probs) else min(len(it["options"]) - 1, max(0, round(ans.score)))
    return it["options"][top], probs, float(ans.score)


ask_item(e12_items[0])                                    # warm-up (not timed)
e12_rows, t0 = [], time.perf_counter()
for it in e12_items:
    pick, probs, expected = ask_item(it)
    row = {**it, "pick": pick, "probs": probs, "ok": pick == it["answer"]}
    if it["type"] == "score":
        row["expected_level"] = expected
        row["level_error"] = abs(it["options"].index(pick) - it["answer_index"])
    e12_rows.append(row)
e12_ms = (time.perf_counter() - t0) * 1000 / len(e12_items)
print(f"E12: {len(e12_rows)} items, {e12_ms:.0f} ms per question\n")

print(f"{'type':8s} | {'items':>5} | {'correct':>9}")
for t in ["noul", "choice", "score", "ALL"]:
    rows = [r for r in e12_rows if t == "ALL" or r["type"] == t]
    print(f"{t:8s} | {len(rows):>5} | {sum(r['ok'] for r in rows):>4} / {len(rows):<4}")
sc = [r for r in e12_rows if r["type"] == "score"]
print(f"score: exact level {sum(r['level_error'] == 0 for r in sc)} / {len(sc)} | within 1 level {sum(r['level_error'] <= 1 for r in sc)} / {len(sc)} | "
      f"mean |expected - true| {sum(abs(r['expected_level'] - r['answer_index']) for r in sc) / len(sc):.2f} levels")
yes_rows = [r for r in e12_rows if r["type"] == "noul"]
print(f"yes / no: answered yes {sum(r['pick'] == 'yes' for r in yes_rows)} of {len(yes_rows)} (truth: {sum(r['answer'] == 'yes' for r in yes_rows)} yes)")

print("\nper category (all types):")
for cat in sorted({r["category"] for r in e12_rows}):
    rows = [r for r in e12_rows if r["category"] == cat]
    print(f"  {cat:10s} {sum(r['ok'] for r in rows):>3} / {len(rows)}")

groups = {}
for r in e12_rows:
    if "pair" in r:
        groups.setdefault(r["pair"], []).append(r)
print(f"\nquestion groups (same text, different questions): all answers right in {sum(all(r['ok'] for r in g) for g in groups.values())} of {len(groups)} groups")
for pid, g in groups.items():
    print(f"  {pid}: " + " | ".join(f"{r['question'][:34]} -> {r['pick']}{'' if r['ok'] else ' x'}" for r in g))

print("\nmistakes:")
for r in [r for r in e12_rows if not r["ok"]]:
    s_ = r["state"] if len(r["state"]) <= 46 else r["state"][:43] + "..."
    print(f"  {r['id']} {r['type']:6s} | {s_:46s} | {r['question'][:38]:38s} | expected {r['answer']:14s} | got {r['pick']}")

with open("e12_decider.json", "w") as fh:
    json.dump({"ms_per_question": e12_ms, "rows": [{k: r[k] for k in ("id", "type", "category", "pick", "ok", "probs")} for r in e12_rows]}, fh)


## 11 — Latency, measured the same way as our models

Each question is timed on its own, warm, with the GPU synchronized before and after, prompt building and tokenization included. Reported as p50 / p95 / mean milliseconds per question on the T4, for the same tests as `lora_combined.ipynb`: E10 (every user turn, full history and 3-turn window), E11, banking and E12. It runs after sections 1–3, and uses `ask_item` from section 10 (run section 10 first, or its definitions cell).


In [ ]:
# Section 11 — Decider latency per question: p50 / p95 / mean (ms)
GPU = torch.device(DECIDER_DEVICE)


def timed(fn):
    torch.cuda.synchronize(GPU)
    t0 = time.perf_counter()
    fn()
    torch.cuda.synchronize(GPU)
    return (time.perf_counter() - t0) * 1000


def summary(ms):
    ms = sorted(ms)
    return f"p50 {ms[len(ms) // 2]:.0f} ms | p95 {ms[min(len(ms) - 1, int(0.95 * len(ms)))]:.0f} ms | mean {sum(ms) / len(ms):.0f} ms ({len(ms)} questions)"


decide([{"role": "user", "content": "warm-up"}], BANKING_AGENTS)          # warm-up (not timed)
lat = {"E10 full": [], "E10 window 3": [], "E11": [], "banking": [], "E12": []}
for conv in real_convs:
    for u in conv["user_turns"]:
        prefix = conv["messages"][:u["message_index"] + 1]
        lat["E10 full"].append(timed(lambda: decide(prefix, conv["agents"])))
        lat["E10 window 3"].append(timed(lambda: decide(prefix, conv["agents"], window=3)))
for m in ft["messages"]:
    lat["E11"].append(timed(lambda: decide([{"role": "user", "content": m["message"]}], ft["companies"][m["company"]])))
for ex in banking_all:
    lat["banking"].append(timed(lambda: decide(ex["messages"], BANKING_AGENTS)))
for it in e12_items:
    lat["E12"].append(timed(lambda: ask_item(it)))
print("Strands decider 2B latency (T4, one question at a time):")
for k, v in lat.items():
    print(f"  {k:13s}: {summary(v)}")
with open("decider_latency.json", "w") as fh:
    json.dump({k: v for k, v in lat.items()}, fh)

## 12 — Independent test set (written outside the project)

The same three files as `03_lora_signpost.ipynb` (`data/modernbert_router/independent/`), converted by `build_independent_test_data.py`:
- **questions** (300): yes / no, choice and score, asked in the decider's native question types, as in section 10;
- **routing** (40 conversations, 172 turns): every user turn routed with the history so far, full and with a 3-turn window. The decider has no current-agent input, so there is no live / given distinction;
- **first turn** (200): one message, 5 agents.

Flagged items are reported separately (see `build_independent_test_data.py`). Questions: `claim_check` (22 "does the text contradict the claim" yes / no items, all answering "yes"). The 11 V1 "score" items that were not one low → high scale were corrected in the converter (3 reordered, 8 retyped as choice; no answer changed). Routing: `vague_start` / `label_conflict` (V1: 1 turn). Run after sections 1–3 and section 10 (for `ask_item`).


In [ ]:
# Section 12 — the independent test set
ind_q = json.load(open(DATA_DIR / "independent" / "questions.json"))["items"]
ind_r = json.load(open(DATA_DIR / "independent" / "routing.json"))
ind_f = json.load(open(DATA_DIR / "independent" / "first_turn.json"))["messages"]
GPU = torch.device(DECIDER_DEVICE)
TURN_TYPES = ["start", "answer", "follow", "aspect", "closing", "also_stay", "switch", "short_switch", "return", "handoff_ok", "branch"]


def timed_call(fn):
    torch.cuda.synchronize(GPU); t0 = time.perf_counter(); out = fn(); torch.cuda.synchronize(GPU)
    return out, (time.perf_counter() - t0) * 1000


def lat(ms):
    ms = sorted(ms)
    return f"p50 {ms[len(ms) // 2]:.0f} ms | p95 {ms[min(len(ms) - 1, int(0.95 * len(ms)))]:.0f} ms"


# questions
rows, ms_q = [], []
for it in ind_q:
    (pick, probs, expected), ms = timed_call(lambda: ask_item(it))
    ms_q.append(ms)
    rows.append({**it, "pick": pick, "ok": pick == it["answer"],
                 "level_error": abs(it["options"].index(pick) - it["answer_index"]) if it["type"] == "score" else None})
by = {t: sum(r["ok"] for r in rows if r["type"] == t) for t in ["noul", "choice", "score"]}
yn = [r for r in rows if r["type"] == "noul"]
yes_ok = sum(r["ok"] for r in yn if r["answer"] == "yes")
sc = [r for r in rows if r["type"] == "score"]
print(f"IND questions | all {sum(r['ok'] for r in rows)}/{len(rows)} | yes/no {by['noul']}/{len(yn)} (truth yes: {yes_ok}/"
      f"{sum(r['answer'] == 'yes' for r in yn)}, truth no: {by['noul'] - yes_ok}/{sum(r['answer'] == 'no' for r in yn)}) | "
      f"choice {by['choice']}/{sum(r['type'] == 'choice' for r in rows)} | score {by['score']}/{len(sc)} "
      f"(within 1 level {sum(r['level_error'] <= 1 for r in sc)}) | {lat(ms_q)}")
clean = [r for r in rows if not r.get("flag")]
claim = [r for r in rows if r.get("flag") == "claim_check"]
sc_ok = [r for r in sc if r.get("flag") != "unordered_scale"]
print(f"  without flagged items: {sum(r['ok'] for r in clean)}/{len(clean)} | claim-check yes/no (all truth 'yes'): "
      f"{sum(r['ok'] for r in claim)}/{len(claim)} | score on proper scales: {sum(r['ok'] for r in sc_ok)}/{len(sc_ok)} "
      f"(within 1 level {sum(r['level_error'] <= 1 for r in sc_ok)}) | unordered 'scales': "
      f"{sum(r['ok'] for r in sc if r.get('flag') == 'unordered_scale')}/{len(sc) - len(sc_ok)}")

# routing (stateless: history only)
for window in [None, 3]:
    rr, ms_r = [], []
    for conv in ind_r:
        for u in conv["user_turns"]:
            prefix = conv["messages"][:u["message_index"] + 1]
            (pick, _), ms = timed_call(lambda: decide(prefix, conv["agents"], window=window))
            ms_r.append(ms)
            rr.append({"type": u["turn_type"], "flag": u["flag"], "ok": pick == u["expected_agent"]})
    clean = [r for r in rr if not r["flag"]]
    print(f"IND routing (decider, {'full history' if window is None else 'window 3'}) | all {sum(r['ok'] for r in rr)}/{len(rr)} | "
          f"unflagged turns {sum(r['ok'] for r in clean)}/{len(clean)} | "
          + " ".join(f"{t} {sum(r['ok'] for r in rr if r['type'] == t)}/{sum(r['type'] == t for r in rr)}" for t in TURN_TYPES if any(r['type'] == t for r in rr))
          + " | flagged: " + " ".join(f"{f} {sum(r['ok'] for r in rr if r['flag'] == f)}/{sum(r['flag'] == f for r in rr)}" for f in ["vague_start", "label_conflict"])
          + f" | {lat(ms_r)}")

# first turn
fr, ms_f = [], []
for m in ind_f:
    (pick, probs), ms = timed_call(lambda: decide([{"role": "user", "content": m["message"]}], m["agents"]))
    ms_f.append(ms)
    fr.append({"type": m["type"], "ok": pick == m["target_agent"], "top2": m["target_agent"] in list(probs)[:2]})
print(f"IND first turn | top-1 {sum(r['ok'] for r in fr)}/{len(fr)} | top-2 {sum(r['top2'] for r in fr)}/{len(fr)} | "
      + " ".join(f"{k} {sum(r['ok'] for r in fr if r['type'] == k)}/{sum(r['type'] == k for r in fr)}" for k in sorted({r['type'] for r in fr}))
      + f" | {lat(ms_f)}")

---
## 13 — What we learned about the decider

The decider's results, from `experiments/modernbert_router_ledger.md` (benchmark B1, runs 14, 19, 22). Signpost's side is in `03`.

**Where it is strong**
- **Grounded questions:** E12 160 / 170 and independent 253 / 300. It's best on **score** questions (independent 70 / 92 against Signpost's 43), a general reader trained on about 30 public classification datasets.
- **First messages:** E11 96 / 100 (top-2 100), independent 178 / 200, level with Signpost.

**Where it is weak**
- **Switches inside a conversation.** It has no current-agent input, so every turn is routed blind. E10 real switches: 10 / 20 at the test companies (13 with a window). Independent: short switches 2 / 20 (6 with a window) and switches 18 / 46 (21), on cases written outside the project, so this isn't an artefact of our own test writing.
- **History length.** With the full history, older topics pull its answer back: E10 151 / 193. A window of the last 3 user turns lifts it to 163 (answers +6, switches +4), but returns to an earlier topic get worse (6 → 3 of 7) because the topic falls out of the window. We tried the same window on our router (`02`, run 10b): no gain, because we pass the current agent.
- **Double negation.** On the independent set's 22 "does the text contradict the claim …" items it answered 0 / 22 (as did our model). These are reported separately.

**Caveats on the comparison**
- **Banking isn't unseen for it:** banking77 and CLINC-OOS are in its training data.
- **Its latency is an upper bound.** It ran in bfloat16 on a T4, which has no fast bfloat16 path, and without its optional kernels (`flash-linear-attention`, `causal-conv1d` fell back to plain PyTorch). Even so: p50 185–439 ms against Signpost's ~50–70 ms. Costs also scale differently: the decider's with prompt length (history + all options in one prompt), Signpost's with the number of options (one pass each, batched).
- **It's zero-shot and general-purpose;** Signpost was trained for exactly these question types, on data written by the same author as most of our tests. The independent set is the fairest comparison.